In [1]:
import numpy as np
from datetime import date, timedelta, datetime
import matplotlib.pyplot as plt
import data_processing as dp
#import machine_learning as ml
import pandas as pd
#import torch
misure = ["Cobb angle","Riss sign","Rib hump", "Frontal imbalance","ATR","Trace"]
osservazioni = ["Diagnosi","Fisioterapia","Corsetto","Corsetto:ore_gg",
                "Average SRS","SRS: Function average","SRS: Mental Health average","SRS: Pain average","SRS: Assessment of self-image average"]

In [ ]:
list_pazienti = dp.get_all_data(99999) #99999 se li vuoi tutti
#Li trasformo in qualcosa di leggibile da MATLAB
matrice = dp.create_lista_generica(list_pazienti, misure, osservazioni)
matrice = dp.inclusion_criteria(matrice)
matrice.to_csv('clean_matrice.csv', index=False)

In [ ]:
def has_forbidden_sequence(group,col):
    fisioterapia = group[col].values
    # Cerca la sequenza [0, 1, 0, 1] in ordine consecutivo
    for i in range(len(fisioterapia) - 3):
        if fisioterapia[i] == 0 and fisioterapia[i+1] == 1 and fisioterapia[i+2] == 0 and fisioterapia[i+3] == 1:
            return True
    return False

#DA MODIFICARE!!
matrice = pd.read_csv("/home/ae/Documents/Synced/Università/Magistrale/Mecatronica/OHDSI/clean_matrice.csv")
print(f"Altezza matrice: {len(matrice)}")

#Rimuovi corsetti non specificati
cols = ['3661733', '4040386', '45762560', '2000000041', '2000000044', '2000000053']
matrice_no_corsetti_strani = matrice.groupby('id').filter(
    lambda g: ~((g[cols] == 1).any(axis=1).any())
)
matrice_no_corsetti_strani.drop(columns=cols, inplace=True)
print(f"Altezza matrice senza corsetti strani: {len(matrice_no_corsetti_strani)}")

#Rimuovi i maschi
matrice_femmine = matrice_no_corsetti_strani[matrice_no_corsetti_strani["gender"]==1]
print(f"Altezza matrice femmine: {len(matrice_femmine)}")

#Rimuovi dati mancanti a Cobb
matrice_solo_con_cobb = matrice_femmine[pd.notna(matrice_femmine["Cobb angle"])]
print(f"Altezza matrice con cobb: {len(matrice_solo_con_cobb)}")

#Rimuovi pazienti con prima visita >=15 anni
matrice_prima_visita = matrice_solo_con_cobb.groupby('id').filter(lambda x: x['eta'].iloc[0] < 15)
print(f"Altezza matrice prima visita: {len(matrice_prima_visita)}")

#Rimuovi pazienti con una diagnosi non corretta 
#Alcuni pazienti han diagnosi più particolari, non le guardiamo
matrice_diagnosi = matrice_prima_visita.groupby('id').filter(
    lambda g: ((g["Diagnosi"] == 1).any())
)
matrice_diagnosi.drop(columns="Diagnosi",inplace=True)
print(f"Altezza matrice matrice_diagnosi: {len(matrice_diagnosi)}")

#Rimuovi pazienti che hanno <= 4 visite
matrice_n_visite = matrice_diagnosi.groupby('id').filter(
    lambda g: len(g)>4)
print(f"Altezza matrice n_visite: {len(matrice_n_visite)}")

#Rimuovi i corsetti rigidi e flessibili
#Lavoriamo solo con i very rigid
cols = ['Rigid_Brace','Flexible_Brace']
matrice_no_corsetti_medi = matrice_n_visite.groupby('id').filter(
    lambda g: ~((g[cols] == 1).any(axis=1).any())
)
matrice_no_corsetti_medi.drop(columns=cols, inplace=True)
print(f"Altezza matrice senza corsetti strani: {len(matrice_no_corsetti_medi)}")

#Rimuovi pazienti che non hanno mai fatto fisioterapia
matrice_no_fisioterapia = matrice_no_corsetti_medi.groupby('id').filter(
    lambda g: ~((g['Fisioterapia'] == 0).all())
)
print(f"Altezza matrice con almeno 1 fisioterapia: {len(matrice_no_fisioterapia)}")

#Rimuovi pazienti che non hanno mai messo il corsetto
matrice_no_brace = matrice_no_fisioterapia.groupby('id').filter(
    lambda g: ~((g['Very_Rigid_Brace'] == 0).all())
)
print(f"Altezza matrice con almeno 1 brace: {len(matrice_no_brace)}")

#Rimuovi pazienti con una sequenza non concorde
#Le pazienti devono avere 
#No Brace->Brace->No Brace
#Stessa cosa con la fisioterapia
matrice_fisioterapia_stoppata_ritornata = matrice_no_brace.groupby('id').filter(
    lambda g: not has_forbidden_sequence(g,"Fisioterapia")
)
print(f"Altezza matrice con seq fisio: {len(matrice_fisioterapia_stoppata_ritornata)}")

matrice_corsetto_stoppata_ritornata = matrice_fisioterapia_stoppata_ritornata.groupby('id').filter(
    lambda g: not has_forbidden_sequence(g,"No_Brace")
)
print(f"Altezza matrice con seq corsetto: {len(matrice_corsetto_stoppata_ritornata)}")
matrice_corsetto_stoppata_ritornata.to_csv('matrice_half_filtered.csv', index=False)

Altezza matrice: 157787
Altezza matrice senza corsetti strani: 157325
Altezza matrice femmine: 114205
Altezza matrice con cobb: 52992
Altezza matrice prima visita: 47014
Altezza matrice matrice_diagnosi: 25266
Altezza matrice n_visite: 15905
Altezza matrice senza corsetti strani: 7550
Altezza matrice con almeno 1 fisioterapia: 7411
Altezza matrice con almeno 1 brace: 5903
Altezza matrice con seq fisio: 5480
Altezza matrice con seq corsetto: 5395


In [ ]:
#X_labels = ["eta","Cobb angle", "Riss sign", "Rib hump", "Fisioterapia","Corsetto:ore_gg","No_Brace","Rigid_Brace","Very_Rigid_Brace","Flexible_Brace", "Frontal imbalance","ATR","Trace"]
#y_labels = ["Average SRS","SRS: Function average","SRS: Mental Health average",
#            "SRS: Pain average","SRS: Assessment of self-image average"]
#y_labels = ["Average SRS"]
#binarie = ["Fisioterapia","No_Brace","Rigid_Brace","Very_Rigid_Brace","Flexible_Brace"]